In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
    'torch>=2.3.0', 'numpy>=1.26.0', 'soundfile>=0.12.1',
], check=True)

In [ ]:
import os, json, time, threading
from pathlib import Path
from datetime import datetime
import yaml, requests
import numpy as np
import torch
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
SYNTH_DIR       = WORK_DIR / 'synthesized_audio'
STREAMS_DIR     = WORK_DIR / 'streams'
MANIFEST_PATH   = WORK_DIR / 'e2e_episodes.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p5b.json'
CONFIG_DIR      = Path('/kaggle/input/urdu-asr-pipelines/config')

STREAMS_DIR.mkdir(parents=True, exist_ok=True)

N_CODEBOOKS   = 8
K_TEXT        = 0
K_AGENT_SEM   = 1
K_AGENT_AC    = slice(2, 9)
K_USER_SEM    = 9
K_USER_AC     = slice(10, 17)
STREAM_WIDTH  = 17
SAVE_EVERY    = 100

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        s = {k: c.get_secret(k) for k in ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY','ANTHROPIC_API_KEY']}
        print('[secrets] Kaggle'); return s
    except Exception: pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv; load_dotenv(env_file)
    required = ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY','ANTHROPIC_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing: raise RuntimeError(f'Missing: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS      = load_secrets()
HF_TOKEN     = SECRETS['HF_TOKEN_PRIMARY']
with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE45_REPO = repos_cfg['repos']['stage45_e2e']['repo_id']
HF_API       = HfApi(token=HF_TOKEN)
print(f'[config] stage45: {STAGE45_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — interleaved={state["stats"]["interleaved"]}')
            return state
        except Exception: pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE45_REPO}/resolve/main/checkpoint_p5b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f: json.dump(state, f)
            print(f'[checkpoint] HF fallback — interleaved={state["stats"]["interleaved"]}')
            return state
    except Exception: pass
    print('[checkpoint] fresh start')
    return {'done_ids': [], 'stats': {'interleaved': 0, 'failed': 0}, 'last_updated': None}

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p5b.json', repo_id=STAGE45_REPO,
                repo_type='dataset', commit_message='p5b checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state    = load_checkpoint()
done_set = set(state['done_ids'])

In [ ]:
def build_interleaved_stream(episode, tokenizer):
    user_turns  = [t for t in episode['turns'] if t['speaker'] == 'user'  and t.get('audio_token_path')]
    agent_turns = [t for t in episode['turns'] if t['speaker'] == 'agent' and t.get('audio_token_path')]

    if not user_turns or not agent_turns:
        return None, 'missing_turns'

    def load_tokens(token_path):
        local = SYNTH_DIR / Path(token_path).name
        if local.exists():
            return np.load(str(local))
        raise FileNotFoundError(f'Token file not found: {local}')

    user_token_arrays  = [load_tokens(t['audio_token_path']) for t in user_turns]
    agent_token_arrays = [load_tokens(t['audio_token_path']) for t in agent_turns]

    user_tokens  = np.concatenate(user_token_arrays,  axis=0)
    agent_tokens = np.concatenate(agent_token_arrays, axis=0)

    T = min(len(user_tokens), len(agent_tokens))
    if T < 10:
        return None, 'too_short'

    user_tokens  = user_tokens[:T]
    agent_tokens = agent_tokens[:T]

    all_text = ' '.join(
        t.get('transcript_urdu', '')
        for t in episode['turns']
        if t['speaker'] == 'agent' and t.get('transcript_urdu')
    )
    text_ids = tokenizer.Encode(all_text) if all_text.strip() else [0]
    text_ids_padded = np.array(
        text_ids[:T] + [0] * max(0, T - len(text_ids)),
        dtype=np.int32
    )[:T]

    stream = np.zeros((T, STREAM_WIDTH), dtype=np.int32)
    stream[:, K_TEXT]      = text_ids_padded
    stream[:, K_AGENT_SEM] = agent_tokens[:, 0]
    stream[:, K_AGENT_AC]  = agent_tokens[:, 1:8]
    stream[:, K_USER_SEM]  = user_tokens[:, 0]
    stream[:, K_USER_AC]   = user_tokens[:, 1:8]

    return stream, 'ok'


import sentencepiece as spm
tok = spm.SentencePieceProcessor()
tok.Load(str(WORK_DIR / 'model_checkpoint' / 'tokenizer.model'))
print('[interleave] tokenizer loaded')

In [ ]:
updated_ep_path = WORK_DIR / 'agent_episodes_with_audio.jsonl'
if not updated_ep_path.exists():
    raise FileNotFoundError('agent_episodes_with_audio.jsonl not found — run p5a first')

all_episodes = []
with open(updated_ep_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: all_episodes.append(json.loads(line))

pending = [ep for ep in all_episodes if ep['episode_id'] not in done_set]
print(f'[p5b] total={len(all_episodes)} done={len(done_set)} pending={len(pending)}')

for idx, episode in enumerate(pending):
    ep_id = episode['episode_id']

    try:
        stream, status = build_interleaved_stream(episode, tok)

        if stream is None:
            print(f'  [skip] {ep_id}: {status}')
            with cp_lock:
                state['stats']['failed'] += 1
                done_set.add(ep_id)
                state['done_ids'].append(ep_id)
            continue

        stream_path = STREAMS_DIR / f'{ep_id}.npy'
        np.save(str(stream_path), stream)

        manifest_record = {
            'episode_id':       ep_id,
            'domain':           episode.get('domain'),
            'difficulty':       episode.get('difficulty'),
            'outcome':          episode.get('outcome'),
            'stream_path':      f'streams/{ep_id}.npy',
            'stream_shape':     list(stream.shape),
            'tools_used':       episode.get('tools_used', []),
            'tool_chain_type':  episode.get('tool_chain_type'),
            'split':            episode.get('split', 'train'),
            'generated_at':     datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        }

        with open(MANIFEST_PATH, 'a', encoding='utf-8') as f:
            f.write(json.dumps(manifest_record, ensure_ascii=False) + '\n')

        with cp_lock:
            state['stats']['interleaved'] += 1
            done_set.add(ep_id)
            state['done_ids'].append(ep_id)

    except Exception as e:
        print(f'  [error] {ep_id}: {e}')
        with cp_lock:
            state['stats']['failed'] += 1
            done_set.add(ep_id)
            state['done_ids'].append(ep_id)

    if (idx + 1) % SAVE_EVERY == 0 or idx + 1 == len(pending):
        save_checkpoint(state, upload=(idx+1) % (SAVE_EVERY*5) == 0)
        print(f'  [{idx+1}/{len(pending)}] interleaved={state["stats"]["interleaved"]} failed={state["stats"]["failed"]}')

save_checkpoint(state, upload=True)
total_streams = sum(1 for _ in STREAMS_DIR.glob('*.npy'))
print(f'\n[p5b] {total_streams} stream files in {STREAMS_DIR}')
print('[done] ready for p5c_upload.ipynb')